In [2]:
import sys
sys.path.append("/host/d/Github/")
import os
import numpy as np
import pandas as pd
import nibabel as nb
import re
import json
import shutil
import Osteosarcoma.functions_collection as ff 
import Osteosarcoma.Data_processing as Data_processing

### Patient split
already done, saved in labels_with_image_info_included_5fold

### Prepare data for nnunet_raw

In [ ]:
# patient_list = pd.read_excel('/host/d/Data/Habitats/Jishuitan/Patient_lists/labels_with_image_info_included_5fold.xlsx')
# print(patient_list.shape)


# save_folder = '/host/d/Data/Habitats/Jishuitan/nnUNet_raw/Dataset601_TumorFirstSet'
# ff.make_folder([save_folder, os.path.join(save_folder, 'imagesTr'), os.path.join(save_folder, 'imagesTs'), os.path.join(save_folder, 'labelsTr')])

# for i in range(0, len(patient_list)):
#     patient_index = patient_list.iloc[i]['Patient_index']
#     patient_split = patient_list.iloc[i]['fold']

#     # set it is tr or ts
#     if patient_split <4:
#         phase = 'Tr'
#     else:
#         phase = 'Ts'

#     # find the image data
#     img_file = os.path.join('/host/d/Data/Habitats/Jishuitan/original_data',str(patient_index),'img.nii.gz')

#     # copy image data
#     img_save_path = os.path.join(save_folder, 'images' + phase, 'TumorFirstSet_' + str(patient_index).zfill(4) + '_0000.nii.gz')
#     shutil.copyfile(img_file, img_save_path)

#     # find the mask data
#     if phase == 'Tr':
#         mask_file = os.path.join('/host/d/Data/Habitats/Jishuitan/original_data',str(patient_index),'label.nii.gz')

#         mask_save_path = os.path.join(save_folder, 'labels' + phase, 'TumorFirstSet_' + str(patient_index).zfill(4) + '.nii.gz')
#         shutil.copyfile(mask_file, mask_save_path)


(81, 23)


In [4]:
patient_list_set1 = pd.read_excel('/host/d/Data/Habitats/Jishuitan/Patient_lists/labels_with_image_info_included_set1.xlsx')
patient_set_list_set1 = patient_list_set1['Patient_set']
patient_index_list_set1 = patient_list_set1['Patient_index']
phase_list_set1 = ['Tr'] * len(patient_list_set1)
patient_list_set2 = pd.read_excel('/host/d/Data/Habitats/Jishuitan/Patient_lists/image_info_set2.xlsx')
patient_set_list_set2 = patient_list_set2['Patient_set']
patient_index_list_set2 = patient_list_set2['Patient_index']
phase_list_set2 = ['Tr' if patient_list_set2.iloc[i]['Have_seg'] == 'Yes' else 'Ts' for i in range(len(patient_list_set2))]


# concatenate 
patient_set_list = pd.concat([patient_set_list_set1, patient_set_list_set2], ignore_index=True)
patient_index_list = pd.concat([patient_index_list_set1, patient_index_list_set2], ignore_index=True)
phase_list = pd.concat([pd.Series(phase_list_set1), pd.Series(phase_list_set2)], ignore_index=True)


save_folder = '/host/d/Data/Habitats/Jishuitan/nnUNet_raw/Dataset602_Tumor'
ff.make_folder([save_folder, os.path.join(save_folder, 'imagesTr'), os.path.join(save_folder, 'imagesTs'), os.path.join(save_folder, 'labelsTr')])

for i in range(0, len(patient_set_list)):
    patient_set = patient_set_list.iloc[i]
    patient_index = patient_index_list.iloc[i]
    phase = phase_list.iloc[i]


    # find the image data
    img_file = os.path.join('/host/d/Data/Habitats/Jishuitan/original_data',patient_set, str(patient_index),'img.nii.gz')

    # copy image data
    img_save_path = os.path.join(save_folder, 'images' + phase, 'Tumor_'+patient_set +'_' + str(patient_index).zfill(4) + '_0000.nii.gz')
    if os.path.isfile(img_save_path) == False:
        shutil.copyfile(img_file, img_save_path)

    # find the mask data
    if phase == 'Tr':
        mask_file = os.path.join('/host/d/Data/Habitats/Jishuitan/original_data',patient_set, str(patient_index),'label.nii.gz')

        mask_save_path = os.path.join(save_folder, 'labels' + phase, 'Tumor_'+patient_set +'_' + str(patient_index).zfill(4) + '.nii.gz')
        shutil.copyfile(mask_file, mask_save_path)


### write the json file

In [5]:
# write the json file
save_folder = '/host/d/Data/Habitats/Jishuitan/nnUNet_raw/Dataset602_Tumor'
json_example = os.path.join(save_folder, 'dataset_raw.json')
with open(json_example, 'r') as file:
    data = json.load(file)

# Now 'data' is a Python dictionary or list containing the JSON data
print(data)

{'name': 'Tumor', 'licence': 'CC-BY-SA 4.0', 'relase': '1.0 04/05/2018', 'tensorImageSize': '3D', 'channel_names': {'0': 'CT'}, 'labels': {'background': 0, 'tumor': 1}, 'numTraining': 60, 'numTest': 40, 'file_ending': '.nii.gz', 'training': [{'image': './imagesTr/hippocampus_367.nii.gz', 'label': './labelsTr/hippocampus_367.nii.gz'}, {'image': './imagesTr/hippocampus_304.nii.gz', 'label': './labelsTr/hippocampus_304.nii.gz'}, {'image': './imagesTr/hippocampus_204.nii.gz', 'label': './labelsTr/hippocampus_204.nii.gz'}, {'image': './imagesTr/hippocampus_279.nii.gz', 'label': './labelsTr/hippocampus_279.nii.gz'}, {'image': './imagesTr/hippocampus_308.nii.gz', 'label': './labelsTr/hippocampus_308.nii.gz'}, {'image': './imagesTr/hippocampus_375.nii.gz', 'label': './labelsTr/hippocampus_375.nii.gz'}, {'image': './imagesTr/hippocampus_216.nii.gz', 'label': './labelsTr/hippocampus_216.nii.gz'}, {'image': './imagesTr/hippocampus_316.nii.gz', 'label': './labelsTr/hippocampus_316.nii.gz'}, {'imag

In [6]:


train_list = []
test_list = []
for i in range(0,len(patient_set_list)):
    patient_set = patient_set_list.iloc[i]
    patient_index = patient_index_list.iloc[i]
    patient_name = 'Tumor_'+patient_set +'_' + str(patient_index).zfill(4)
    phase = phase_list.iloc[i]

    if phase == 'Tr':
        train_list.append({'image': "./images%s/%s_0000.nii.gz" % (phase, patient_name), 'label': "./labels%s/%s.nii.gz" % (phase, patient_name)})
    else:
        phase = 'Ts'
        test_list.append("./images%s/%s_0000.nii.gz" % (phase, patient_name))

data["training"] = train_list
data["test"] = test_list
data["numTraining"] = len(train_list)
data["numTest"] = len(test_list)

save_json_file = os.path.join(save_folder, 'dataset.json')
with open(save_json_file, 'w') as file:
    json.dump(data, file, indent=4)